# Tensor Tests

Unit tests for `Tensor` construction, arithmetic, reshape, and transpose.

## Setup

In [10]:
import numpy as np
import traceback

# ── import your Tensor class here ──────────────────────────────────────────
from Tensor import Tensor

# ── helpers ────────────────────────────────────────────────────────────────
def make(data, **kwargs):
    """Wrap a list or ndarray in a Tensor."""
    return Tensor(np.array(data, **kwargs))

def assert_close(t, expected, rtol=1e-6):
    np.testing.assert_allclose(t.data, np.array(expected), rtol=rtol)

# ── tiny test runner ────────────────────────────────────────────────────────
_results = []

def run(name, fn):
    try:
        fn()
        _results.append(("PASS", name))
        print(f"  ✅  {name}")
    except Exception as e:
        _results.append(("FAIL", name, str(e)))
        print(f"  ❌  {name}")
        traceback.print_exc()

def summary():
    passed = sum(1 for r in _results if r[0] == "PASS")
    total  = len(_results)
    print(f"\n{'─'*50}")
    print(f"  {passed}/{total} tests passed")
    if passed == total:
        print("  🎉 All tests passed!")
    print(f"{'─'*50}")


## 1 — Construction

In [3]:
print("── Construction ──────────────────────────────────")

def test_wraps_1d_array():
    arr = np.array([1.0, 2.0, 3.0])
    t = Tensor(arr)
    np.testing.assert_array_equal(t.data, arr)

def test_wraps_2d_array():
    arr = np.array([[1, 2], [3, 4]])
    t = Tensor(arr)
    np.testing.assert_array_equal(t.data, arr)

def test_shape():
    t = make([[1, 2, 3], [4, 5, 6]])
    assert t.shape == (2, 3), f"expected (2,3), got {t.shape}"

def test_size():
    t = make([[1, 2, 3], [4, 5, 6]])
    assert t.size == 6, f"expected 6, got {t.size}"

def test_dtype_float32():
    t = Tensor(np.array([1.0, 2.0], dtype=np.float32))
    assert t.dtype == np.float32, f"expected float32, got {t.dtype}"

def test_dtype_int64():
    t = Tensor(np.array([1, 2], dtype=np.int64))
    assert t.dtype == np.int64, f"expected int64, got {t.dtype}"

def test_scalar_shape():
    t = Tensor(np.array(5.0))
    assert t.shape == (), f"expected (), got {t.shape}"

def test_scalar_size():
    t = Tensor(np.array(5.0))
    assert t.size == 1, f"expected 1, got {t.size}"

def test_data_is_ndarray():
    t = Tensor(np.ones((3, 3)))
    assert isinstance(t.data, np.ndarray)

for name, fn in [
    ("wraps 1-D array",         test_wraps_1d_array),
    ("wraps 2-D array",         test_wraps_2d_array),
    ("shape attribute",         test_shape),
    ("size attribute",          test_size),
    ("dtype float32",           test_dtype_float32),
    ("dtype int64",             test_dtype_int64),
    ("scalar shape == ()",      test_scalar_shape),
    ("scalar size == 1",        test_scalar_size),
    ("data is ndarray",         test_data_is_ndarray),
]:
    run(name, fn)


── Construction ──────────────────────────────────
  ✅  wraps 1-D array
  ✅  wraps 2-D array
  ✅  shape attribute
  ✅  size attribute
  ✅  dtype float32
  ✅  dtype int64
  ✅  scalar shape == ()
  ✅  scalar size == 1
  ✅  data is ndarray


## 2 — Arithmetic: Tensor + Tensor

In [4]:
print("── Tensor + Tensor ───────────────────────────────")

def test_add():
    assert_close(make([1,2,3]) + make([4,5,6]), [5,7,9])

def test_sub():
    assert_close(make([10,20,30]) - make([1,2,3]), [9,18,27])

def test_mul():
    assert_close(make([2,3,4]) * make([5,6,7]), [10,18,28])

def test_truediv():
    assert_close(make([10.,20.,30.]) / make([2.,4.,5.]), [5.,5.,6.])

def test_result_is_tensor():
    result = make([1,2]) + make([3,4])
    assert isinstance(result, Tensor), f"expected Tensor, got {type(result)}"

def test_2d_add():
    a = make([[1,2],[3,4]])
    b = make([[10,20],[30,40]])
    assert_close(a + b, [[11,22],[33,44]])

for name, fn in [
    ("add  [1,2,3]+[4,5,6]",    test_add),
    ("sub  [10,20,30]-[1,2,3]", test_sub),
    ("mul  [2,3,4]*[5,6,7]",    test_mul),
    ("div  [10,20,30]/[2,4,5]", test_truediv),
    ("result is Tensor",         test_result_is_tensor),
    ("2-D add",                  test_2d_add),
]:
    run(name, fn)


── Tensor + Tensor ───────────────────────────────
  ✅  add  [1,2,3]+[4,5,6]
  ✅  sub  [10,20,30]-[1,2,3]
  ✅  mul  [2,3,4]*[5,6,7]
  ✅  div  [10,20,30]/[2,4,5]
  ✅  result is Tensor
  ✅  2-D add


## 3 — Arithmetic: Tensor + scalar

In [5]:
print("── Tensor + scalar ───────────────────────────────")

def test_add_int():
    assert_close(make([1,2,3]) + 10, [11,12,13])

def test_add_float():
    assert_close(make([1.,2.]) + 0.5, [1.5,2.5])

def test_sub_scalar():
    assert_close(make([5,6,7]) - 3, [2,3,4])

def test_mul_scalar():
    assert_close(make([2,3,4]) * 3, [6,9,12])

def test_div_scalar():
    assert_close(make([6.,9.,12.]) / 3., [2.,3.,4.])

def test_scalar_result_is_tensor():
    assert isinstance(make([1,2,3]) + 1, Tensor)

def test_mul_zero():
    assert_close(make([1,2,3]) * 0, [0,0,0])

def test_div_one():
    assert_close(make([7.,14.]) / 1., [7.,14.])

for name, fn in [
    ("add int scalar",    test_add_int),
    ("add float scalar",  test_add_float),
    ("sub scalar",        test_sub_scalar),
    ("mul scalar",        test_mul_scalar),
    ("div scalar",        test_div_scalar),
    ("result is Tensor",  test_scalar_result_is_tensor),
    ("mul by zero",       test_mul_zero),
    ("div by one",        test_div_one),
]:
    run(name, fn)


── Tensor + scalar ───────────────────────────────
  ✅  add int scalar
  ✅  add float scalar
  ✅  sub scalar
  ✅  mul scalar
  ✅  div scalar
  ✅  result is Tensor
  ✅  mul by zero
  ✅  div by one


## 4 — Broadcasting

In [6]:
print("── Broadcasting ──────────────────────────────────")

def test_row_broadcast():
    mat = make([[1,2,3],[4,5,6]])     # (2,3)
    row = make([[10,20,30]])          # (1,3)
    assert_close(mat + row, [[11,22,33],[14,25,36]])

def test_col_broadcast():
    mat = make([[1,2],[3,4],[5,6]])   # (3,2)
    col = make([[10],[20],[30]])      # (3,1)
    assert_close(mat + col, [[11,12],[23,24],[35,36]])

def test_3d_broadcast():
    a = make(np.ones((2,3,4)))
    b = make(np.ones((3,4)) * 2)
    r = a * b
    assert r.shape == (2,3,4), f"expected (2,3,4), got {r.shape}"
    assert_close(r, np.full((2,3,4), 2.))

def test_incompatible_raises():
    a = make([[1,2,3]])   # (1,3)
    b = make([[1,2]])     # (1,2)
    try:
        _ = a + b
        assert False, "expected an exception for incompatible shapes"
    except Exception:
        pass   # correct

for name, fn in [
    ("row (1,3) → matrix (2,3)",     test_row_broadcast),
    ("col (3,1) → matrix (3,2)",     test_col_broadcast),
    ("3-D broadcast (3,4)→(2,3,4)",  test_3d_broadcast),
    ("incompatible shapes raise",    test_incompatible_raises),
]:
    run(name, fn)


── Broadcasting ──────────────────────────────────
  ✅  row (1,3) → matrix (2,3)
  ✅  col (3,1) → matrix (3,2)
  ✅  3-D broadcast (3,4)→(2,3,4)
  ✅  incompatible shapes raise


## 5 — reshape

In [7]:
print("── reshape ────────────────────────────────────────")

def test_basic_reshape():
    assert make([1,2,3,4,5,6]).reshape(2,3).shape == (2,3)

def test_reshape_values():
    assert_close(make([1,2,3,4,5,6]).reshape(2,3), [[1,2,3],[4,5,6]])

def test_reshape_to_1d():
    assert make([[1,2],[3,4]]).reshape(4).shape == (4,)

def test_reshape_is_tensor():
    assert isinstance(make([1,2,3,4]).reshape(2,2), Tensor)

def test_neg1_rows():
    assert make([1,2,3,4,5,6]).reshape(-1,3).shape == (2,3)

def test_neg1_cols():
    assert make([1,2,3,4,5,6]).reshape(2,-1).shape == (2,3)

def test_neg1_3d():
    assert Tensor(np.arange(24)).reshape(2,3,-1).shape == (2,3,4)

def test_mismatch_raises_large():
    try:
        make([1,2,3,4,5,6]).reshape(2,4)   # 8 ≠ 6
        assert False, "expected exception"
    except Exception:
        pass

def test_mismatch_raises_small():
    try:
        make([1,2,3,4]).reshape(3,3)        # 9 ≠ 4
        assert False, "expected exception"
    except Exception:
        pass

def test_3d_to_2d():
    t = Tensor(np.arange(24).reshape(2,3,4))
    assert t.reshape(6,4).shape == (6,4)

def test_same_shape():
    t = make([[1,2],[3,4]])
    r = t.reshape(2,2)
    assert r.shape == (2,2)
    assert_close(r, [[1,2],[3,4]])

for name, fn in [
    ("basic reshape → (2,3)",        test_basic_reshape),
    ("reshape preserves values",     test_reshape_values),
    ("reshape to 1-D",               test_reshape_to_1d),
    ("result is Tensor",             test_reshape_is_tensor),
    ("-1 infers rows",               test_neg1_rows),
    ("-1 infers cols",               test_neg1_cols),
    ("-1 in 3-D",                    test_neg1_3d),
    ("size mismatch (too large)",    test_mismatch_raises_large),
    ("size mismatch (too small)",    test_mismatch_raises_small),
    ("3-D → 2-D",                    test_3d_to_2d),
    ("same shape no-op",             test_same_shape),
]:
    run(name, fn)


── reshape ────────────────────────────────────────
  ✅  basic reshape → (2,3)
  ✅  reshape preserves values
  ✅  reshape to 1-D
  ✅  result is Tensor
  ✅  -1 infers rows
  ✅  -1 infers cols
  ✅  -1 in 3-D
  ✅  size mismatch (too large)
  ✅  size mismatch (too small)
  ✅  3-D → 2-D
  ✅  same shape no-op


## 6 — transpose

In [8]:
print("── transpose ─────────────────────────────────────")

def test_2d_shape():
    assert make([[1,2,3],[4,5,6]]).transpose().shape == (3,2)

def test_2d_values():
    assert_close(make([[1,2],[3,4]]).transpose(), [[1,3],[2,4]])

def test_result_is_tensor():
    assert isinstance(make([[1,2],[3,4]]).transpose(), Tensor)

def test_3d_swaps_last_two():
    assert Tensor(np.arange(24).reshape(2,3,4)).transpose().shape == (2,4,3)

def test_3d_values():
    arr = np.arange(24).reshape(2,3,4)
    r   = Tensor(arr).transpose()
    np.testing.assert_array_equal(r.data, np.swapaxes(arr,-1,-2))

def test_double_transpose_identity():
    t = make([[1,2,3],[4,5,6]])
    assert_close(t.transpose().transpose(), t.data)

def test_square_matrix():
    r = make([[1,2],[3,4]]).transpose()
    assert r.shape == (2,2)
    assert_close(r, [[1,3],[2,4]])

for name, fn in [
    ("2-D default shape (3,2)",       test_2d_shape),
    ("2-D values",                    test_2d_values),
    ("result is Tensor",              test_result_is_tensor),
    ("3-D swaps last two axes",       test_3d_swaps_last_two),
    ("3-D values match swapaxes",     test_3d_values),
    ("double-transpose == identity",  test_double_transpose_identity),
    ("square matrix",                 test_square_matrix),
]:
    run(name, fn)


── transpose ─────────────────────────────────────
  ✅  2-D default shape (3,2)
  ✅  2-D values
  ✅  result is Tensor
  ✅  3-D swaps last two axes
  ✅  3-D values match swapaxes
  ✅  double-transpose == identity
  ✅  square matrix


## Summary

In [9]:
summary()


──────────────────────────────────────────────────
  45/45 tests passed
  🎉 All tests passed!
──────────────────────────────────────────────────
